# 05 — KDB, KLB, dan kepatuhan zonasi

**KDB** (Koefisien Dasar Bangunan) = luas tapak bangunan ÷ luas persil.
**KLB** (Koefisien Lantai Bangunan) = total luas lantai ÷ luas persil.

Batas persil tidak tersedia sebagai data terbuka, jadi kita memakai **blok kota** (dari notebook 04) sebagai
pengganti persil. Batas zonasi di notebook ini **sintetis** — dibuat masuk akal untuk latihan, *bukan* RDTR
Kota Semarang yang sebenarnya.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import numpy as np
import pandas as pd

from kota import CFG, PROC

blok = gpd.read_file(PROC / "blok.gpkg", layer="blok")
kel = gpd.read_file(PROC / "morfologi.gpkg", layer="kelurahan")
bangunan = gpd.read_file(PROC / "bangunan_utm.gpkg", layer="bangunan")
bangunan = bangunan[bangunan["tapak_m2"] >= 10].reset_index(drop=True)  # buang poligon sisa digitasi
blok["blok_id"] = np.arange(len(blok))
print(len(blok), "blok |", len(bangunan), "bangunan")

## 1. Tapak bangunan per blok
Potong bangunan dengan batas blok (`overlay`). Bangunan yang terpotong garis blok adalah **masalah data**
(jalan melintas bangunan, atau digitasi yang meleset) — bukan temuan pelanggaran.

In [ ]:
pot = gpd.overlay(bangunan[["id", "building", "building:levels", "height", "tapak_m2", "geometry"]],
                  blok[["blok_id", "geometry"]], how="intersection", keep_geom_type=True)
pot["bagian_m2"] = pot.area
lintas = pot.groupby("id")["blok_id"].nunique()
print("Bangunan di lebih dari satu blok:", int((lintas > 1).sum()),
      f"({(lintas > 1).mean():.1%} dari bangunan yang masuk blok)")

In [ ]:
jumlah = pot.groupby("blok_id")["bagian_m2"].sum() / blok.set_index("blok_id")["luas_m2"]
print("Blok dengan KDB > 1 bila tapak dijumlah langsung:", int((jumlah > 1).sum()))

KDB di atas 1 mustahil: tapak lebih luas dari bloknya. Penyebabnya poligon bangunan OSM yang **bertumpuk**
(digitasi ganda, atau `building:part` di atas bangunan induk). Gabungkan (*dissolve*) tapak per blok dulu,
sehingga luas yang tumpang-tindih hanya dihitung sekali.

In [ ]:
tapak = pot[["blok_id", "geometry"]].dissolve(by="blok_id").area
blok["tapak_m2"] = blok["blok_id"].map(tapak).fillna(0)
blok["kdb"] = blok["tapak_m2"] / blok["luas_m2"]
blok["kdb"].describe().round(3)

## 2. Zonasi sintetis
Aturan pembentuk zona (hanya untuk latihan):
- tipe bangunan dominan `industrial`/`warehouse` → **I** (industri),
- dominan `commercial`/`retail`/`office` → **K** (perdagangan dan jasa),
- selain itu perumahan, dibagi menurut kepadatan bangunan kelurahan: **R-2** tinggi, **R-3** sedang, **R-4** rendah.

In [ ]:
ZONA = pd.DataFrame({
    "zona": ["R-2", "R-3", "R-4", "K", "I"],
    "keterangan": ["Perumahan kepadatan tinggi", "Perumahan kepadatan sedang", "Perumahan kepadatan rendah",
                   "Perdagangan dan jasa", "Industri"],
    "kdb_maks": [0.70, 0.60, 0.50, 0.80, 0.60],
    "klb_maks": [2.1, 1.8, 1.0, 4.0, 1.8],
}).set_index("zona")

tipe = pot.assign(t=pot["building"].fillna("yes")).groupby(["blok_id", "t"])["bagian_m2"].sum()
dominan = tipe.groupby(level=0).idxmax().map(lambda x: x[1])
blok["tipe_dominan"] = blok["blok_id"].map(dominan).fillna("kosong")

padat = blok["kel_id"].map(kel.set_index("id")["bangunan_per_ha"])
blok["zona"] = np.select(
    [blok["tipe_dominan"].isin(["industrial", "warehouse"]),
     blok["tipe_dominan"].isin(["commercial", "retail", "office"]),
     padat >= 35, padat >= 15],
    ["I", "K", "R-2", "R-3"], default="R-4")
blok = blok.join(ZONA, on="zona")
blok["zona"].value_counts()

## 3. KDB terhadap batas zonasi
Galat posisi bangunan OSM dan batas blok yang kasar berarti selisih kecil tidak bermakna. Kita pisahkan
**melampaui jelas** (lebih dari 0,10 di atas batas) dari **dalam batas galat** (0 – 0,10 di atas batas).

In [ ]:
TOL = 0.10
blok["selisih_kdb"] = blok["kdb"] - blok["kdb_maks"]
blok["status_kdb"] = np.select([blok["selisih_kdb"] > TOL, blok["selisih_kdb"] > 0],
                               ["melampaui jelas", "dalam batas galat"], default="sesuai")
ringkas = blok.groupby(["zona", "status_kdb"]).size().unstack(fill_value=0)
ringkas["% melampaui jelas"] = (100 * ringkas.get("melampaui jelas", 0) / ringkas.sum(axis=1)).round(1)
ringkas

## 4. Estimasi KLB
KLB butuh jumlah lantai, yang tidak terlihat dari atas. Urutan sumber: `building:levels` → `height` ÷ 3 m →
**asumsi 1 lantai**. Laporkan berapa bagian luas tapak yang hanya berdasar asumsi.

In [ ]:
lv = pd.to_numeric(pot["building:levels"], errors="coerce")
tinggi = pd.to_numeric(pot["height"].astype(str).str.extract(r"([\d.]+)")[0], errors="coerce")
pot["lantai"] = lv.fillna(np.round(tinggi / 3)).clip(lower=1)
pot["sumber_lantai"] = np.where(lv.notna(), "building:levels", np.where(tinggi.notna(), "height", "asumsi"))
pot["lantai"] = pot["lantai"].fillna(1)
print((pot.groupby("sumber_lantai")["bagian_m2"].sum() / pot["bagian_m2"].sum()).round(3))

blok["lantai_m2"] = blok["blok_id"].map((pot["bagian_m2"] * pot["lantai"]).groupby(pot["blok_id"]).sum()).fillna(0)
blok["klb"] = blok["lantai_m2"] / blok["luas_m2"]
blok["klb_melampaui"] = blok["klb"] > blok["klb_maks"] + TOL
print("Blok dengan estimasi KLB melampaui batas (+toleransi):", int(blok["klb_melampaui"].sum()))

## 5. Peta dan batas kesimpulan
Peta ini adalah **alat penyaring**: menunjukkan ke mana petugas perlu melihat. Ia bukan bukti pelanggaran —
blok bukan persil, batas zonasinya sintetis, dan data bangunan OSM punya galat posisi serta kelengkapan.

In [ ]:
warna = {"sesuai": "#d9d9d9", "dalam batas galat": "#fdae61", "melampaui jelas": "#d7191c"}
ax = blok.plot(color=blok["status_kdb"].map(warna), figsize=(9, 7), linewidth=0)
ax.set_axis_off()
ax.set_title(f"KDB blok terhadap batas zonasi sintetis — {CFG['name']}")

In [ ]:
blok.to_file(PROC / "kdb_blok.gpkg", layer="blok")
ZONA.reset_index().to_csv(PROC / "zonasi_sintetis.csv", index=False)
print("disimpan: kdb_blok.gpkg, zonasi_sintetis.csv")